# GAN 3D de cavernas: auditoria, preparação, treino e avaliação no Colab

**Nenhum treinamento foi executado na entrega.** `RUN_TRAINING=False` e
`RUN_EVALUATION=False`. A auditoria de 15/11/2026 encontrou 2, 1, 98 e 5 voxels
de ar nos quatro interiores 64³: o preflight os rejeita. Corrija a reconstrução,
valide sua geometria e gere novos manifestos antes de habilitar o treino.

Leia `docs/plano-redes-neurais.md` e `docs/auditoria/gan/README.md` no pacote.
Partição por caverna, sem recortes atravessando treino/validação/teste; seeds de
inferência não criam novas cavernas independentes. O discriminador recebe só
o volume e distingue referência reconstruída (1) de saída de G (0).

Prepare localmente o ZIP com
`python -m cavegen.training.prepare_colab` usando o código desta revisão.
Abra este notebook em uma VM Colab com GPU. Upload privado; o ZIP contém
voxels derivados e recibos, sem nuvens brutas nem artigos.

In [ ]:
%pip install -q "numpy>=1.26" "scipy>=1.11" "torch>=2.2" "matplotlib>=3.8"

## Carregar o pacote e verificar sua integridade

Faça upload de `gan_colab_bundle.zip`. Se a sessão já tem o pacote extraído,
a célula apenas verifica novamente seu manifesto. Para trocar de versão, use
outra pasta vazia em `ROOT`. Não é necessário instalar todas as dependências
dos algoritmos clássicos para executar o pipeline neural.

In [ ]:
from pathlib import Path, PurePosixPath
import hashlib, json, os, sys, zipfile

ROOT = Path("/content/cavegan")
if not (ROOT / "bundle_manifest.json").exists():
    from google.colab import files # type: ignore
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.endswith(".zip")]
    assert len(zip_names) == 1, "Envie exatamente um ZIP do prepare_colab"
    ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_names[0]) as archive:
        for name in archive.namelist():
            parts = PurePosixPath(name)
            assert not parts.is_absolute() and ".." not in parts.parts and "\\" not in name
        archive.extractall(ROOT)

manifest = json.loads((ROOT / "bundle_manifest.json").read_text(encoding="utf-8"))
for name, expected in manifest.items():
    assert hashlib.sha256((ROOT / name).read_bytes()).hexdigest() == expected, name
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print(f"{len(manifest)} arquivos verificados; código em {ROOT}")

## Ambiente e persistência

Opcionalmente monte seu Drive para checkpoints e relatórios. O tempo máximo
inclui validação; um passo já iniciado pode ultrapassá-lo. A VRAM de um passo
completo só será conhecida quando um experimento for autorizado. Este notebook
não faz uma atualização de pesos durante a preparação.

In [ ]:
import torch, numpy as np
from cavegen.training.gan_training import environment_record

USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive # type: ignore
    drive.mount("/content/drive")
    RUNS = Path("/content/drive/MyDrive/cavegan_runs")
else:
    RUNS = Path("/content/gan_runs")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(json.dumps(environment_record(torch.device(DEVICE)), indent=2))
print("Checkpoints:", RUNS, "| GPU disponível:", torch.cuda.is_available())

## Inventário congelado e partições

Os hashes são portáveis: paths dos voxels são relativos ao diretório de
referências. Paths Linux históricos nos recibos descrevem a origem e não são
usados para abrir arquivos no Colab. O CSV de origem preserva as pendências
de licença; não publique os derivados como se fossem de domínio público.

In [ ]:
from cavegen.training.reference_data import verify_splits
from IPython.display import display, Image

REFERENCES = ROOT / "data/processed/references"
AUDIT = json.loads((ROOT / "docs/auditoria/gan/reference_audit.json").read_text(encoding="utf-8"))
SPLITS = json.loads((ROOT / "docs/auditoria/gan/splits.json").read_text(encoding="utf-8"))
FOLD = "holdout_indian_tunnel"
rows = verify_splits(SPLITS, REFERENCES, FOLD)
for row in rows:
    print(row["split"], row["group"], row["path"])
print("Estado dos dados:", SPLITS["data_status"])
for record in AUDIT["records"]:
    print(record["group"], record["kind"], "voxels True:", record["true_voxels"], record["issues"])
if (ROOT / "docs/auditoria/gan/projections.png").exists():
    display(Image(filename=str(ROOT / "docs/auditoria/gan/projections.png")))

## Configuração e preflight sem treino

Escolha `occupancy64.json` ou `sdf64.json`. `allow_candidate_proxy=True` explicita
o experimento com ar inferido e não libera interiores sem geometria resolvida.
Nas referências desta entrega o preflight deve falhar e manter o treino bloqueado.
As duas variantes usam resolução nativa 64³; não há redução oculta para 16³.

In [ ]:
from cavegen.training.gan_training import TrainingConfig, preflight

CONFIG_FILE = "configs/gan/occupancy64.json"
config = TrainingConfig.from_dict(json.loads((ROOT / CONFIG_FILE).read_text(encoding="utf-8")))
DATA_READY = False
try:
    print(json.dumps(preflight(REFERENCES, SPLITS, FOLD, config), indent=2))
    DATA_READY = True
except ValueError as error:
    print("TREINO BLOQUEADO:", error)
print("Configuração efetiva:", config.to_dict())

## Inspecionar arquiteturas (opcional; somente forward)

Esta célula mostra parâmetros e shapes com pesos não treinados. Não é uma
avaliação de qualidade nem mede o pico de memória de backward/R1/Adam.
O ensaio de memória de um passo completo fica para depois da aprovação dos dados.

In [ ]:
RUN_MODEL_CHECK = False
if RUN_MODEL_CHECK:
    from cavegen.training.gan_models import Generator3D, Discriminator3D, latent_for_seed
    G = Generator3D(config.model).to(DEVICE).eval()
    D = Discriminator3D(config.model).to(DEVICE).eval()
    with torch.no_grad():
        field = G(latent_for_seed(0, config.model.latent_dim).to(DEVICE))
        print("G:", tuple(field.shape), "D:", tuple(D(field).shape))
    print("Parâmetros G/D:", sum(p.numel() for p in G.parameters()), sum(p.numel() for p in D.parameters()))
    del G, D, field
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

## Treino explícito ou retomada — desativado

Depois de corrigir os dados, repetir auditoria e preflight e revisar o protocolo,
habilite a célula conscientemente. Para as repetições propostas, use seeds de
treino 17, 29 e 43 em pastas separadas, mantendo as mesmas seeds de inferência.
Para cada fold o teste permanece fora da seleção; `best.pt` é escolhido só na
validação. `latest.pt` permite retomada com modelos, Adam e RNG torch.

Queda de sessão: monte Drive antes do treino. Em erro de memória/NaN a sessão
para; a retomada usa o último checkpoint de um passo completo. Troca de GPU ou
versão é registrada e pode alterar resultados; não há promessa de igualdade
bit a bit entre plataformas. Se o limite de tempo ocorrer antes da primeira
validação, haverá `latest.pt`, mas ainda não haverá `best.pt`.

In [ ]:
from cavegen.training.gan_training import train

RUN_TRAINING = False
RESUME = False
RUN_DIR = RUNS / f"gan_{config.model.representation}_64" / FOLD / f"train_seed_{config.seed}"
if RUN_TRAINING:
    assert DATA_READY, "Corrija os dados e execute novamente o preflight"
    train(REFERENCES, SPLITS, FOLD, config, RUN_DIR, resume=RESUME, device=DEVICE)
else:
    print("Nenhum treino iniciado. Diretório proposto:", RUN_DIR)

## Avaliação final — seeds arbitrárias, sem ajuste no teste

O domínio aceito é inteiro de 0 a 2⁶³−1. Não é viável enumerar todas as seeds.
O protocolo inicial usa 0..255 e três extremos; a lista pode ser ampliada sem
treinar novamente. Seeds 1.000.000..1.000.015 são reservadas à validação no
protocolo padrão; sobreposições serão registradas se você pedir outras listas.

O relatório preserva todos os casos, inclusive vazios, duplicados e desconexos.
Inclui os controles de cópia e todo-sólido. As métricas são descritivas com uma
caverna de teste por fold; muitas seeds não resolvem a escassez de fontes.

In [ ]:
from cavegen.training.evaluate_gan import evaluate_checkpoint, evaluation_seeds
from cavegen.training.reference_data import canonical_hash

RUN_EVALUATION = False
TEST_SEEDS = evaluation_seeds(explicit=list(range(256)) + [2**16-1, 2**32-1, 2**63-1])
CHECKPOINT = RUN_DIR / "best.pt"
EVAL_DIR = RUN_DIR / ("test_" + canonical_hash(TEST_SEEDS)[:12])
if RUN_EVALUATION:
    assert CHECKPOINT.exists(), "Ainda não existe checkpoint validado"
    report = evaluate_checkpoint(CHECKPOINT, REFERENCES, SPLITS, TEST_SEEDS, EVAL_DIR, device=DEVICE)
    print(json.dumps({key: report[key] for key in ("generated_count", "degenerate_fraction",
          "duplicate_fraction", "normalized_wasserstein", "pattern_2cube_js_nats")}, indent=2))
else:
    print("Avaliação desativada;", len(TEST_SEEDS), "seeds previstas")

## Inspeção e exportação após a avaliação

Cada seed gera `seed_N.npz` (`data`, booleano `True=ar`, e metadados) e
`seed_N.field.npz` (campo contínuo sem reparo). A variante SDF inclui MAE contra
a recodificação do próprio volume gerado. Preserve esses arquivos, `run.json`,
`status.json`, logs JSONL e checkpoints junto com o relatório. A biblioteca
`GAN3DGenerator` aceita qualquer seed válida para inferência futura.

In [ ]:
if RUN_EVALUATION:
    import matplotlib.pyplot as plt
    from cavegen.datastream.voxel_io import load_volume_npz
    volume = load_volume_npz(EVAL_DIR / f"seed_{TEST_SEEDS[0]}.npz")
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for axis, ax in enumerate(axes):
        ax.imshow(volume.data.any(axis=axis), origin="lower", cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"Projeção de ar; eixo reduzido={axis}")
    plt.show()